# Curriculum Fine-tuning: CRNN for Government Documents

3-phase curriculum to avoid catastrophic forgetting:
- Phase 1 (5 epochs):  100% Mozhi — refresh base model memory
- Phase 2 (10 epochs): 50% Mozhi + 50% govdoc — balanced mixing
- Phase 3 (10 epochs): 20% Mozhi + 80% govdoc — focus on govdoc

**Expected time:** ~35 minutes on T4
**Expected CER improvement:** 0.89 → 0.45–0.60

In [ ]:
# Cell 1: Mount Drive and setup
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, zipfile

REPO        = '/content/aksharDrishti'
DRIVE       = '/content/drive/MyDrive/AksharDrishti'
DRIVE_W     = DRIVE + '/weights'
DRIVE_DATA  = DRIVE + '/datasets'

if not os.path.exists(REPO):
    !git clone https://github.com/Ayush-04-spec/akshardrishti.git {REPO}
else:
    !cd {REPO} && git pull
print('Repo ready')

In [ ]:
# Cell 2: Copy weights, charset, extract datasets
os.makedirs(REPO + '/weights/recognize', exist_ok=True)

# Weights
shutil.copy(DRIVE_W + '/recognize/crnn_mozhi_himr.pt',  REPO + '/weights/recognize/crnn_mozhi_himr.pt')
shutil.copy(DRIVE_W + '/recognize/charset_deva.txt',    REPO + '/weights/recognize/charset_deva.txt')
print('Weights copied')

# govdoc_finetune dataset
gd_zip = DRIVE_DATA + '/govdoc_finetune.zip'
gd_dst = REPO + '/datasets/govdoc_finetune'
if os.path.exists(gd_dst): shutil.rmtree(gd_dst)
with zipfile.ZipFile(gd_zip, 'r') as z: z.extractall(REPO + '/datasets/')
n_gd = len(os.listdir(gd_dst + '/images'))
print(f'govdoc images: {n_gd}')

# mozhi_sample dataset
ms_zip = DRIVE_DATA + '/mozhi_sample.zip'
ms_dst = REPO + '/datasets/mozhi_sample'
if os.path.exists(ms_dst): shutil.rmtree(ms_dst)
with zipfile.ZipFile(ms_zip, 'r') as z: z.extractall(REPO + '/datasets/')
n_ms = len(os.listdir(ms_dst + '/images'))
print(f'mozhi_sample images: {n_ms}')

In [ ]:
# Cell 3: Imports and helpers
import sys, csv, random, time
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from pathlib import Path
sys.path.insert(0, REPO)

from akshardrishti.recognize.crnn_model import Charset, CRNN, greedy_decode
from train.dataset import LineImageDataset, ctc_collate, load_index
from akshardrishti.postprocess.text_repair import normalize_for_scoring

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

charset = Charset.load(Path(REPO + '/weights/recognize/charset_deva.txt'))
print('Charset size:', len(charset))

def fix_paths(rows, images_dir):
    for r in rows:
        fname = r['image_path'].replace('\\', '/').split('/')[-1]
        r['image_path'] = images_dir + '/' + fname
    return rows

def ed(a, b):
    if a==b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev=list(range(len(b)+1))
    for ca in a:
        cur=[prev[0]+1]
        for j,cb in enumerate(b,1): cur.append(min(prev[j]+1,cur[j-1]+1,prev[j-1]+(ca!=cb)))
        prev=cur
    return prev[-1]

def corpus_cer(preds, refs):
    err=chars=0
    for p,r in zip(preds,refs):
        pn=normalize_for_scoring(p); rn=normalize_for_scoring(r)
        err+=ed(pn,rn); chars+=len(rn)
    return err/chars if chars else 0

def eval_cer(model, rows, batch=64):
    ds = LineImageDataset(rows, charset, 32, 512, augment=False)
    dl = DataLoader(ds, batch_size=batch, shuffle=False, num_workers=2, collate_fn=ctc_collate)
    preds=[]; model.eval()
    with torch.no_grad():
        for imgs,_,_,_,_ in dl:
            preds.extend(t for t,_ in greedy_decode(model(imgs.to(DEVICE)), charset))
    return corpus_cer(preds, [r['text'] for r in rows])

def make_loader(rows, batch=32):
    ds = LineImageDataset(rows, charset, 32, 512, augment=True)
    return DataLoader(ds, batch_size=batch, shuffle=True, num_workers=2,
                      collate_fn=ctc_collate, pin_memory=True)

def train_phase(model, optimizer, scheduler, loader, epochs, phase_name, govdoc_rows):
    ctc = nn.CTCLoss(blank=charset.BLANK, zero_infinity=True)
    best_cer = float('inf')
    best_state = None
    for ep in range(1, epochs+1):
        model.train(); total_loss=0; t0=time.perf_counter()
        for imgs, targets, _, tgt_lens, _ in loader:
            imgs=imgs.to(DEVICE); targets=targets.to(DEVICE)
            lp=model(imgs); T,N,C=lp.shape
            loss=ctc(lp, targets, torch.full((N,),T,dtype=torch.long), tgt_lens.to(DEVICE))
            optimizer.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step(); total_loss+=loss.item()
        if scheduler: scheduler.step()
        cer = eval_cer(model, govdoc_rows)
        elapsed = time.perf_counter()-t0
        print(f'[{phase_name}] Epoch {ep}/{epochs}  loss={total_loss/len(loader):.4f}  govdoc_CER={cer:.4f}  ({elapsed:.0f}s)')
        if cer < best_cer:
            best_cer=cer; best_state={
                k:v.cpu().clone() for k,v in model.state_dict().items()}
            print(f'  -> New best {best_cer:.4f}')
    if best_state:
        model.load_state_dict(best_state)
    return best_cer

In [ ]:
# Cell 4: Load datasets
random.seed(42)

# govdoc rows
gd_rows = load_index(Path(REPO + '/datasets/govdoc_finetune/index.csv'))
gd_rows = fix_paths(gd_rows, REPO + '/datasets/govdoc_finetune/images')
gd_rows = [r for r in gd_rows if charset.coverage(r['text']) >= 0.8]
ok_gd = sum(1 for r in gd_rows if Path(r['image_path']).exists())
print(f'govdoc: {len(gd_rows)} rows, {ok_gd} images found')

# mozhi_sample rows
ms_rows = load_index(Path(REPO + '/datasets/mozhi_sample/index_train.csv'))
ms_rows = fix_paths(ms_rows, REPO + '/datasets/mozhi_sample/images')
ms_rows = [r for r in ms_rows if charset.coverage(r['text']) >= 0.999]
ok_ms = sum(1 for r in ms_rows if Path(r['image_path']).exists())
print(f'mozhi_sample: {len(ms_rows)} rows, {ok_ms} images found')

In [ ]:
# Cell 5: Load base model
import pathlib
WEIGHTS = REPO + '/weights/recognize/crnn_mozhi_himr.pt'
try:
    ckpt = torch.load(WEIGHTS, map_location='cpu', weights_only=True)
except:
    ckpt = torch.load(WEIGHTS, map_location='cpu', weights_only=False)

model = CRNN(num_classes=len(charset)).to(DEVICE)
key = 'model' if 'model' in ckpt else 'model_state_dict'
model.load_state_dict(ckpt[key])
print(f'Base model loaded (key={key})')
print(f'Baseline govdoc CER: {eval_cer(model, gd_rows[:200]):.4f}')

In [ ]:
# Cell 6: PHASE 1 — 100% Mozhi, 5 epochs (refresh base memory)
print('='*50)
print('PHASE 1: 100% Mozhi (5 epochs, LR=1e-4)')
print('='*50)

p1_rows = random.sample(ms_rows, min(2000, len(ms_rows)))
random.shuffle(p1_rows)
loader1  = make_loader(p1_rows)
opt1     = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
sched1   = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=5)

cer1 = train_phase(model, opt1, sched1, loader1, 5, 'P1', gd_rows)
print(f'Phase 1 best govdoc CER: {cer1:.4f}')

In [ ]:
# Cell 7: PHASE 2 — 50% Mozhi + 50% govdoc, 10 epochs
print('='*50)
print('PHASE 2: 50% Mozhi + 50% govdoc (10 epochs, LR=5e-5)')
print('='*50)

n = len(gd_rows)
p2_rows = gd_rows + random.sample(ms_rows, min(n, len(ms_rows)))
random.shuffle(p2_rows)
loader2 = make_loader(p2_rows)
opt2    = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
sched2  = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=10)

cer2 = train_phase(model, opt2, sched2, loader2, 10, 'P2', gd_rows)
print(f'Phase 2 best govdoc CER: {cer2:.4f}')

In [ ]:
# Cell 8: PHASE 3 — 80% govdoc + 20% Mozhi, 10 epochs
print('='*50)
print('PHASE 3: 80% govdoc + 20% Mozhi (10 epochs, LR=2e-5)')
print('='*50)

n_moz = max(100, len(gd_rows) // 4)
p3_rows = gd_rows + random.sample(ms_rows, min(n_moz, len(ms_rows)))
random.shuffle(p3_rows)
loader3 = make_loader(p3_rows)
opt3    = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=1e-4)
sched3  = torch.optim.lr_scheduler.CosineAnnealingLR(opt3, T_max=10)

cer3 = train_phase(model, opt3, sched3, loader3, 10, 'P3', gd_rows)
print(f'Phase 3 best govdoc CER: {cer3:.4f}')

print()
print('=== CURRICULUM SUMMARY ===')
print(f'Phase 1 best: {cer1:.4f}')
print(f'Phase 2 best: {cer2:.4f}')
print(f'Phase 3 best: {cer3:.4f}')

In [ ]:
# Cell 9: Save to Drive
OUT = REPO + '/weights/recognize/crnn_curriculum_ft.pt'
torch.save({'model_state_dict': model.state_dict(),
            'charset_size': len(charset),
            'phase1_cer': cer1, 'phase2_cer': cer2, 'phase3_cer': cer3},
           OUT)
shutil.copy(OUT, DRIVE_W + '/recognize/crnn_curriculum_ft.pt')
print(f'Saved to Drive: {DRIVE_W}/recognize/crnn_curriculum_ft.pt')
print(f'Download and place at: weights/recognize/crnn_curriculum_ft.pt')